In [1]:
import os
import hopsworks
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import time

d:\Virtual_Environments\10pearls_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

project = hopsworks.login(
    api_key_value=os.getenv("AQI_Predictor_KEY"),
    project="Pearls_AQI_Predict",
    cert_folder="./hopsworks-certs"
)

fs = project.get_feature_store()
fg = fs.get_feature_group(name="aqi_features_multan", version=1)
df = fg.read()

2026-08-24 18:27:32,415 INFO: Initializing external client
2026-08-24 18:27:32,417 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-24 18:27:37,195 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41139
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (24.12s) 


In [3]:
print(df.shape)
df.head()

(26232, 45)


,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,cloud_cover,precipitation,us_aqi,pm2_5,pm10,...,pressure_lag_12h,aqi_lag_24h,wind_speed_lag_24h,pressure_lag_24h,aqi_roll_mean_6h,aqi_roll_mean_24h,aqi_roll_std_24h,aqi_change_1h,aqi_change_6h,target_aqi_72h
0,2026-02-07 22:00:00+00:00,16.8,68,4.4,1001.0,0,0.0,170,131.8,148.5,...,1003.8,190.0,7.7,1001.2,167.833333,171.708333,6.603551,1.0,4.0,128.0
1,2025-05-28 22:00:00+00:00,36.5,26,5.9,979.4,0,0.0,149,78.7,149.5,...,983.9,153.0,8.1,981.0,160.333333,149.500000,10.193775,3.0,-12.0,156.0
2,2024-04-11 18:00:00+00:00,31.7,29,6.9,992.2,69,0.0,143,26.5,51.5,...,993.7,113.0,8.7,990.2,107.833333,93.041667,15.493278,15.0,55.0,108.0
3,2024-10-09 03:00:00+00:00,24.0,80,6.6,994.3,0,0.0,132,24.3,41.9,...,994.5,133.0,7.4,995.2,132.833333,129.500000,2.859006,-1.0,3.0,151.0
4,2026-01-08 03:00:00+00:00,7.5,88,6.6,1005.6,0,0.0,292,217.0,224.0,...,1005.6,259.0,1.9,1005.4,294.333333,287.875000,8.527768,-3.0,0.0,329.0


In [4]:
test_months = 13  # slightly over a year, ensures full seasonal cycle

cutoff_date = df["time"].max() - pd.DateOffset(months=test_months)

train_df = df[df["time"] < cutoff_date].reset_index(drop=True)
test_df = df[df["time"] >= cutoff_date].reset_index(drop=True)

print("Cutoff date:", cutoff_date)
print("Train shape:", train_df.shape, "| range:", train_df["time"].min(), "to", train_df["time"].max())
print("Test shape:", test_df.shape, "| range:", test_df["time"].min(), "to", test_df["time"].max())

Cutoff date: 2025-07-12 23:00:00+00:00
Train shape: (16727, 45) | range: 2023-08-16 00:00:00+00:00 to 2025-07-12 22:00:00+00:00
Test shape: (9505, 45) | range: 2025-07-12 23:00:00+00:00 to 2026-08-12 23:00:00+00:00


In [5]:
feature_cols = [col for col in train_df.columns if col not in ["time", "target_aqi_72h"]]

X_train = train_df[feature_cols]
y_train = train_df["target_aqi_72h"]

X_test = test_df[feature_cols]
y_test = test_df["target_aqi_72h"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print(feature_cols)

X_train shape: (16727, 43)
X_test shape: (9505, 43)
['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'surface_pressure', 'cloud_cover', 'precipitation', 'us_aqi', 'pm2_5', 'pm10', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'dust', 'month', 'hour', 'day', 'day_of_year', 'day_of_week', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'aqi_lag_1h', 'wind_speed_lag_1h', 'pressure_lag_1h', 'aqi_lag_3h', 'wind_speed_lag_3h', 'pressure_lag_3h', 'aqi_lag_6h', 'wind_speed_lag_6h', 'pressure_lag_6h', 'aqi_lag_12h', 'wind_speed_lag_12h', 'pressure_lag_12h', 'aqi_lag_24h', 'wind_speed_lag_24h', 'pressure_lag_24h', 'aqi_roll_mean_6h', 'aqi_roll_mean_24h', 'aqi_roll_std_24h', 'aqi_change_1h', 'aqi_change_6h']


In [6]:
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"Ridge Regression — RMSE: {rmse_ridge:.2f}, MAE: {mae_ridge:.2f}, R²: {r2_ridge:.3f}")

Ridge Regression — RMSE: 30.97, MAE: 25.15, R²: 0.589


In [7]:
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest — RMSE: {rmse_rf:.2f}, MAE: {mae_rf:.2f}, R²: {r2_rf:.3f}")

Random Forest — RMSE: 33.45, MAE: 26.54, R²: 0.520


In [8]:
y_pred_rf_train = rf.predict(X_train)
rmse_rf_train = np.sqrt(mean_squared_error(y_train, y_pred_rf_train))
r2_rf_train = r2_score(y_train, y_pred_rf_train)

print(f"Random Forest — TRAIN RMSE: {rmse_rf_train:.2f}, TRAIN R²: {r2_rf_train:.3f}")
print(f"Random Forest — TEST  RMSE: {rmse_rf:.2f}, TEST  R²: {r2_rf:.3f}")

Random Forest — TRAIN RMSE: 6.34, TRAIN R²: 0.981
Random Forest — TEST  RMSE: 33.45, TEST  R²: 0.520


In [9]:
import pandas as pd

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(10))

month_cos            0.575543
day_of_year          0.102565
day                  0.061330
aqi_roll_mean_24h    0.041389
aqi_lag_24h          0.030902
day_of_week          0.025789
aqi_roll_std_24h     0.014190
us_aqi               0.013900
dust                 0.013553
pressure_lag_24h     0.008520
dtype: float64


In [10]:
rf_v2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,           # reduced from 15
    min_samples_leaf=20,   # forces each leaf to represent a meaningful chunk of data, not noise
    random_state=42,
    n_jobs=-1
)
rf_v2.fit(X_train, y_train)

y_pred_rf_v2 = rf_v2.predict(X_test)
print(f"RF v2 — RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf_v2)):.2f}, "
      f"MAE: {mean_absolute_error(y_test, y_pred_rf_v2):.2f}, "
      f"R²: {r2_score(y_test, y_pred_rf_v2):.3f}")

# Confirm overfitting gap has shrunk
y_pred_rf_v2_train = rf_v2.predict(X_train)
print(f"RF v2 TRAIN R²: {r2_score(y_train, y_pred_rf_v2_train):.3f}")

RF v2 — RMSE: 33.19, MAE: 26.27, R²: 0.528
RF v2 TRAIN R²: 0.838


In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge_scaled = Ridge(alpha=1.0, random_state=42)
ridge_scaled.fit(X_train_scaled, y_train)
y_pred_ridge_scaled = ridge_scaled.predict(X_test_scaled)

print(f"Ridge (scaled) — RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge_scaled)):.2f}, "
      f"MAE: {mean_absolute_error(y_test, y_pred_ridge_scaled):.2f}, "
      f"R²: {r2_score(y_test, y_pred_ridge_scaled):.3f}")

Ridge (scaled) — RMSE: 30.91, MAE: 25.09, R²: 0.590


In [12]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost — RMSE: {rmse_xgb:.2f}, MAE: {mae_xgb:.2f}, R²: {r2_xgb:.3f}")

# Check for overfitting, same diagnostic as before
y_pred_xgb_train = xgb.predict(X_train)
print(f"XGBoost TRAIN R²: {r2_score(y_train, y_pred_xgb_train):.3f}")

XGBoost — RMSE: 31.65, MAE: 25.21, R²: 0.571
XGBoost TRAIN R²: 0.926


In [13]:
tscv = TimeSeriesSplit(n_splits=5)

ridge_param_grid = {"alpha": [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 200.0]}

start = time.time()
ridge_grid = GridSearchCV(
    Ridge(random_state=42),
    ridge_param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
ridge_grid.fit(X_train_scaled, y_train)
print(f"Ridge tuning took {time.time()-start:.1f}s")
print("Best alpha:", ridge_grid.best_params_)
print("Best CV RMSE:", -ridge_grid.best_score_)

Ridge tuning took 9.1s
Best alpha: {'alpha': 10.0}
Best CV RMSE: 28.5626822874929


In [14]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 8, 12],
    "min_samples_leaf": [10, 20, 50]
}

start = time.time()
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)  # RF doesn't need scaled features
print(f"RF tuning took {time.time()-start:.1f}s")
print("Best params:", rf_grid.best_params_)
print("Best CV RMSE:", -rf_grid.best_score_)

RF tuning took 148.0s
Best params: {'max_depth': 12, 'min_samples_leaf': 10, 'n_estimators': 100}
Best CV RMSE: 15.95944847053959


In [15]:
xgb_param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 300, 400]
}

start = time.time()
xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, subsample=0.8, colsample_bytree=0.8),
    xgb_param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
print(f"XGBoost tuning took {time.time()-start:.1f}s")
print("Best params:", xgb_grid.best_params_)
print("Best CV RMSE:", -xgb_grid.best_score_)

XGBoost tuning took 74.9s
Best params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 400}
Best CV RMSE: 10.437078377175263


In [16]:
# Ridge with tuned alpha
ridge_tuned = Ridge(alpha=10.0, random_state=42)
ridge_tuned.fit(X_train_scaled, y_train)
y_pred_ridge_tuned = ridge_tuned.predict(X_test_scaled)
print(f"Ridge (tuned) — Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge_tuned)):.2f}, "
      f"Test R²: {r2_score(y_test, y_pred_ridge_tuned):.3f}")

# Random Forest with tuned params
rf_tuned = RandomForestRegressor(max_depth=12, min_samples_leaf=10, n_estimators=100, random_state=42, n_jobs=-1)
rf_tuned.fit(X_train, y_train)
y_pred_rf_tuned = rf_tuned.predict(X_test)
y_pred_rf_tuned_train = rf_tuned.predict(X_train)
print(f"RF (tuned) — Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned)):.2f}, "
      f"Test R²: {r2_score(y_test, y_pred_rf_tuned):.3f}, "
      f"Train R²: {r2_score(y_train, y_pred_rf_tuned_train):.3f}")

# XGBoost with tuned params
xgb_tuned = XGBRegressor(max_depth=7, learning_rate=0.1, n_estimators=400, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb_tuned.fit(X_train, y_train)
y_pred_xgb_tuned = xgb_tuned.predict(X_test)
y_pred_xgb_tuned_train = xgb_tuned.predict(X_train)
print(f"XGBoost (tuned) — Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb_tuned)):.2f}, "
      f"Test R²: {r2_score(y_test, y_pred_xgb_tuned):.3f}, "
      f"Train R²: {r2_score(y_train, y_pred_xgb_tuned_train):.3f}")

Ridge (tuned) — Test RMSE: 30.89, Test R²: 0.591
RF (tuned) — Test RMSE: 33.49, Test R²: 0.519, Train R²: 0.935
XGBoost (tuned) — Test RMSE: 32.68, Test R²: 0.542, Train R²: 0.998
